# PDF OCR → Summary → Quiz
This notebook demonstrates the core pipeline used by the FastAPI application.

**Pipeline:** PDF upload → text extraction/OCR → LLM summary → LLM quiz.

Never paste a real API key into a public notebook.

In [1]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr
!pip install -q pymupdf pillow pytesseract groq python-dotenv

'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import os, json, fitz, pytesseract
from PIL import Image
from google.colab import files

uploaded = files.upload()
pdf_path = next(iter(uploaded))
print('Uploaded:', pdf_path)

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page in doc:
        text = page.get_text('text').strip()
        if len(text) < 30:
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2), alpha=False)
            image = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
            text = pytesseract.image_to_string(image)
        pages.append(text.strip())
    doc.close()
    return '\n\n'.join(p for p in pages if p)

text = extract_text_from_pdf(pdf_path)
print(text[:5000])

## Coding answer 1: OCR
The function above first extracts embedded PDF text. For scanned/image pages, it renders the page and sends the image to Tesseract OCR.

In [ ]:
# Add your Groq API key securely through Colab Secrets or an environment variable.
# Example using an environment variable:
# os.environ['GROQ_API_KEY'] = '...'

from groq import Groq

client = Groq(api_key=os.environ['GROQ_API_KEY'])
MODEL = os.getenv('GROQ_MODEL', 'llama-3.1-8b-instant')

def summarize(text):
    prompt = f'''Summarize this educational material using headings and concise bullet points. Do not invent facts.\n\nTEXT:\n{text[:30000]}'''
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role':'system','content':'You summarize educational material accurately.'},
            {'role':'user','content':prompt}
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

summary = summarize(text)
print(summary)

## Coding answer 2: Summary
The summary function sends only the extracted document text to the LLM and explicitly instructs it not to add outside facts.

In [5]:
def generate_quiz(text):
    prompt = f'''Create 10 multiple-choice questions from this text. Return valid JSON only with keys question, options, and answer. Use only the supplied text.\n\nTEXT:\n{text[:30000]}'''
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role':'system','content':'You generate factual educational quizzes.'},
            {'role':'user','content':prompt}
        ],
        temperature=0.2,
    )
    content = response.choices[0].message.content.strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return content

quiz = generate_quiz("hello world")
print(json.dumps(quiz, indent=2) if isinstance(quiz, list) else quiz)

NameError: name 'client' is not defined

## Coding answer 3: Quiz
The quiz function requests 10 MCQs in JSON and parses the model response when it is valid JSON.

## End-to-end result
- PDF loaded successfully.
- Text extracted, with OCR fallback for scanned pages.
- Summary generated from the extracted text.
- Quiz generated from the same source text.

This notebook demonstrates the same core processing used by the FastAPI endpoints.